In [1]:
import pandas as pd 

df = pd.read_csv("cleaned_data_normal.csv", index_col=0)

def get_top_alleles_from_largest_studies(df_filtered, top_n=5, verbose=True):
    """
    Find the highest frequency allele in the top N largest studies by sample size.
    """
    
    # Step 1: Get sample size per population (should be constant within each population)
    pop_size = df_filtered.groupby('population').agg({
        'n': 'first',  # Sample size of the study
        'allele': 'count'  # allele variety
    }).reset_index()
    pop_size.columns = ['population', 'sample_size', 'num_alleles_reported']
    
    if verbose:
        print(f"Total populations: {len(pop_size)}")
        print(f"\nSample size statistics:")
        print(pop_size['sample_size'].describe())
    
    # Step 2: Get top N populations by sample size
    top_populations = pop_size.nlargest(top_n, 'sample_size')
    
    if verbose:
        print(f"\n{'='*60}")
        print(f"Top {top_n} largest studies:")
        print(f"{'='*60}")
        for idx, row in top_populations.iterrows():
            print(f"{row['population']:<50} n={row['sample_size']:>6,}  ({row['num_alleles_reported']} alleles)")
    
    # Step 3: Filter data for these top populations
    top_studies_data = df_filtered[df_filtered['population'].isin(top_populations['population'])]
    
    # Step 4: For each population, find the allele(s) with highest frequency
    # Using idxmax to get the index of maximum frequency per population
    highest_freq_idx = top_studies_data.groupby('population')['alleles_over_2n'].idxmax()
    highest_freq_alleles = top_studies_data.loc[highest_freq_idx]
    
    # Step 5: Add sample size info and sort by sample size
    result = highest_freq_alleles.merge(
        top_populations[['population', 'sample_size', 'num_alleles_reported']], 
        on='population'
    ).sort_values('sample_size', ascending=False)
    
    # Rename columns for clarity
    result = result.rename(columns={
        'n': 'sample_size_original',
        'alleles_over_2n': 'frequency'
    })
    
    # Drop duplicate sample_size column if it exists
    if 'sample_size_original' in result.columns and 'sample_size' in result.columns:
        result = result.drop(columns=['sample_size_original'])
    
    if verbose:
        print(f"\n{'='*60}")
        print(f"Highest frequency allele in each of top {top_n} studies:")
        print(f"{'='*60}")
        for idx, row in result.iterrows():
            print(f"\n{row['population']}")
            print(f"  Sample size: {row['sample_size']:,}")
            print(f"  Highest freq allele: {row['gene']}*{row['allele']}")
            print(f"  Frequency: {row['frequency']:.4f} ({row['frequency']*100:.2f}%)")
    
    return result

In [2]:
# Get top alleles from largest studies
result = get_top_alleles_from_largest_studies(df, top_n=100, verbose=False)
print(result.head())

    Unnamed: 0 gene   allele                       population  frequency  \
13       61533    A  A*02:01     Germany DKMS - German donors   0.285204   
69       61748    A  A*02:01      USA NMDP European Caucasian   0.275556   
62       86800    C  C*04:01  USA NMDP African American pop 2   0.203689   
75       61754    A  A*02:01      USA NMDP Mexican or Chicano   0.223083   
78       60724    A  A*01:01      USA NMDP South Asian Indian   0.154578   

   resolution  sample_size  num_alleles_reported  
13    4-digit      3456066                  1143  
69    4-digit      1242890                  1015  
62    4-digit       416581                   646  
75    4-digit       261235                   590  
78    4-digit       185391                   465  


In [3]:
# Most common top allele in top-n studies
print("Most common top allele in top-n studies: ")
print(result["allele"].value_counts())

Most common top allele in top-n studies: 
allele
A*02:01    40
A*24:02    11
A*11:01    10
A*01:01     6
C*04:01     5
C*07:02     1
B*44:03     1
B*49:01     1
A*02:02     1
B*38:01     1
A*03:02     1
A*30:01     1
B*07:02     1
C*01:02     1
Name: count, dtype: int64


In [4]:
# Alternative method: Calculate average frequency across all studies that report each allele
allele_avg_freq = df.groupby('allele').agg({
    'alleles_over_2n': 'mean',  # Average frequency across all studies
    'population': 'count'        # Number of studies reporting this allele
}).reset_index()

allele_avg_freq.columns = ['allele', 'avg_frequency', 'num_studies']

# Sort by highest average frequency
allele_avg_freq = allele_avg_freq.sort_values('avg_frequency', ascending=False)

print("Top alleles by average frequency across all studies:")
print(allele_avg_freq.head(20))

Top alleles by average frequency across all studies:
       allele  avg_frequency  num_studies
50    A*02:01       0.180470           79
2359  C*07:02       0.126092           58
2216  C*04:01       0.120698           58
416   A*24:02       0.117222           79
2358  C*07:01       0.103249           57
0     A*01:01       0.097399           79
336   A*11:01       0.086022           79
2315  C*06:02       0.076612           58
2138  C*03:04       0.076021           58
262   A*03:01       0.070496           79
2074  C*01:02       0.069731           58
844   B*07:02       0.061611           80
1795  B*51:01       0.056013           80
1291  B*35:01       0.052364           80
939   B*08:01       0.050386           79
1626  B*44:03       0.047538           80
2548  C*12:03       0.046283           58
2137  C*03:03       0.045829           58
1486  B*40:01       0.042987           80
1192  B*18:01       0.042388           78


In [5]:
# Check how many alleles each study (population) has
alleles_per_study = df.groupby('population')['allele'].count().reset_index()
alleles_per_study.columns = ['population', 'num_alleles']
alleles_per_study = alleles_per_study.sort_values('num_alleles', ascending=False)

print("Statistics on number of alleles per study:")
print(alleles_per_study['num_alleles'].describe())
print(f"\nTotal number of studies: {len(alleles_per_study)}")
print(f"Studies with >= 100 alleles: {(alleles_per_study['num_alleles'] >= 100).sum()}")
print(f"Studies with < 100 alleles: {(alleles_per_study['num_alleles'] < 100).sum()}")
print(f"\nMinimum alleles in any study: {alleles_per_study['num_alleles'].min()}")
print(f"Maximum alleles in any study: {alleles_per_study['num_alleles'].max()}")

print("\n" + "="*60)
print("Distribution of allele counts:")
print("="*60)
print(alleles_per_study.head(20))

Statistics on number of alleles per study:
count      81.000000
mean      236.012346
std       182.744938
min        41.000000
25%       144.000000
50%       179.000000
75%       245.000000
max      1143.000000
Name: num_alleles, dtype: float64

Total number of studies: 81
Studies with >= 100 alleles: 76
Studies with < 100 alleles: 5

Minimum alleles in any study: 41
Maximum alleles in any study: 1143

Distribution of allele counts:
                                          population  num_alleles
13                      Germany DKMS - German donors         1143
69                       USA NMDP European Caucasian         1015
62                   USA NMDP African American pop 2          646
72       USA NMDP Hispanic South or Central American          594
75                       USA NMDP Mexican or Chicano          590
76  USA NMDP Middle Eastern or North Coast of Africa          511
23                                     Germany pop 8          479
78                       USA NMDP S

In [6]:
def get_top_n_alleles_per_study(df, n=100, verbose=True):
    """
    Get the top N alleles by frequency from each study.
    If a study has fewer than N alleles, return all available alleles.
    
    Parameters:
    -----------
    df : DataFrame
        The dataframe containing allele data
    n : int
        Number of top alleles to get from each study (default: 100)
    verbose : bool
        Whether to print progress information
        
    Returns:
    --------
    DataFrame with top N alleles from each study
    """
    
    all_top_alleles = []
    
    # Get unique populations (studies)
    populations = df['population'].unique()
    
    if verbose:
        print(f"Processing {len(populations)} studies...")
        print(f"Getting top {n} alleles from each study\n")
    
    studies_with_less_than_n = 0
    
    for pop in populations:
        # Get all alleles for this population
        pop_data = df[df['population'] == pop].copy()
        
        # Sort by frequency (alleles_over_2n) in descending order
        pop_data_sorted = pop_data.sort_values('alleles_over_2n', ascending=False)
        
        # Get top N (or all if less than N available)
        num_available = len(pop_data_sorted)
        num_to_take = min(n, num_available)
        
        if num_available < n:
            studies_with_less_than_n += 1
        
        top_alleles = pop_data_sorted.head(num_to_take)
        all_top_alleles.append(top_alleles)
    
    # Combine all results
    result_df = pd.concat(all_top_alleles, ignore_index=True)
    
    if verbose:
        print(f"{'='*60}")
        print(f"Summary:")
        print(f"{'='*60}")
        print(f"Total studies processed: {len(populations)}")
        print(f"Studies with < {n} alleles: {studies_with_less_than_n}")
        print(f"Studies with >= {n} alleles: {len(populations) - studies_with_less_than_n}")
        print(f"Total alleles in result: {len(result_df)}")
        print(f"\nResult shape: {result_df.shape}")
    
    return result_df

In [7]:
# Get top 100 alleles from each study
top_100_per_study = get_top_n_alleles_per_study(df, n=100, verbose=True)
print("\nFirst few rows of result:")
print(top_100_per_study.head(10))

Processing 81 studies...
Getting top 100 alleles from each study

Summary:
Total studies processed: 81
Studies with < 100 alleles: 5
Studies with >= 100 alleles: 76
Total alleles in result: 7913

Result shape: (7913, 7)

First few rows of result:
   Unnamed: 0 gene   allele       population  alleles_over_2n     n resolution
0       67020    A  A*11:01  China Hubei Han         0.261857  3732    4-digit
1       84019    C  C*01:02  China Hubei Han         0.202361  3732    4-digit
2       89048    C  C*07:02  China Hubei Han         0.169151  3732    4-digit
3       68764    A  A*24:02  China Hubei Han         0.159095  3732    4-digit
4       41214    B  B*46:01  China Hubei Han         0.150245  3732    4-digit
5       33607    B  B*40:01  China Hubei Han         0.140942  3732    4-digit
6       62720    A  A*02:07  China Hubei Han         0.128177  3732    4-digit
7       85581    C  C*03:04  China Hubei Han         0.117535  3732    4-digit
8       61492    A  A*02:01  China Hubei H

In [8]:
# List populations represented in the top 100 alleles per study
populations_top_100 = (
    top_100_per_study["population"]
    .value_counts()
    .rename_axis("population")
    .reset_index(name="num_alleles_in_top_100")
)

print(f"Number of populations: {len(populations_top_100)}")
print(populations_top_100.to_string(index=False))

Number of populations: 81
                                                  population  num_alleles_in_top_100
                                             China Hubei Han                     100
                                             USA Asian pop 2                     100
                                     USA NMDP Caribean Black                     100
           USA NMDP American Indian South or Central America                     100
                             USA NMDP Alaska Native or Aleut                     100
                             USA NMDP African American pop 2                     100
                                            USA NMDP African                     100
                                          USA Hispanic pop 2                     100
                                         USA Caucasian pop 4                     100
                                  USA African American pop 4                     100
                                    USA

In [9]:
# NMDP subgroup names to exclude
nmdp_groups = {
    "African American pop 2",
    "African",
    "Black South or Central American",
    "Caribbean Black",
    "Chinese",
    "Filipino",
    "Japanese",
    "Korean",
    "South Asian Indian",
    "Southeast Asian",
    "Vietnamese",
    "Mexican or Chicano",
    "Hispanic South or Central American",
    "Caribbean Hispanic",
    "European Caucasian",
    "Middle Eastern or North Coast of Africa",
    "North American Amerindian",
    "Alaska Native or Aleut",
    "American Indian South or Central America",
    "Hawaiian or other Pacific Islander",
    "Caribbean Indian",
}

# Get population names only, excluding the 21 NMDP groups
other_populations = sorted(
    population
    for population in populations_top_100["population"].unique()
    if not (
        population.startswith("USA NMDP ")
        and population.removeprefix("USA NMDP ") in nmdp_groups
    )
)

print(f"Number of other populations: {len(other_populations)}")
print("\n".join(other_populations))

Number of other populations: 64
China Hubei Han
China Jiangsu Han
China South Han pop 2
China Zhejiang Han
Colombia BogotÃ¡ Cord Blood
Croatia pop 4
Czech Republic NMDR
France French Bone Marrow Donor Registry
Germany DKMS - Austria minority
Germany DKMS - Bosnia and Herzegovina minority
Germany DKMS - China minority
Germany DKMS - Croatia minority
Germany DKMS - France minority
Germany DKMS - German donors
Germany DKMS - Greece minority
Germany DKMS - Italy minority
Germany DKMS - Netherlands minority
Germany DKMS - Portugal minority
Germany DKMS - Romania minority
Germany DKMS - Spain minority
Germany DKMS - Turkey minority
Germany DKMS - United Kingdom minority
Germany pop 6
Germany pop 8
Hong Kong Chinese BMDR
Hong Kong Chinese HKBMDR HLA 11 loci
Hong Kong Chinese cord blood registry
India Tamil Nadu
Ireland Northern
Israel Arab pop 2
Israel Argentina Jews
Israel Ashkenazi Jews pop 3
Israel Bukhara Jews
Israel Druze
Israel Ethiopia Jews
Israel Georgia Jews
Israel Iran Jews
Israel I

In [10]:
# Create a Markdown file listing NMDP-related and non-NMDP studies
populations = sorted(populations_top_100["population"].dropna().unique())

nmdp_studies = [p for p in populations if p.startswith("USA NMDP ")]
non_nmdp_studies = [p for p in populations if not p.startswith("USA NMDP ")]

output_path = "study_populations.md"

with open(output_path, "w", encoding="utf-8") as file:
    file.write("# Study Populations\n\n")

    file.write(f"## NMDP-related studies ({len(nmdp_studies)})\n\n")
    for study in nmdp_studies:
        file.write(f"- {study}\n")

    file.write(f"\n## Non-NMDP studies ({len(non_nmdp_studies)})\n\n")
    for study in non_nmdp_studies:
        file.write(f"- {study}\n")

print(f"Saved study names to {output_path}")

Saved study names to study_populations.md


In [12]:
import os
# Print statistics for the generated Markdown file(s)
md_files = [output_path]

for md_file in md_files:
    if not os.path.exists(md_file):
        print(f"{md_file} not found.")
        continue

    with open(md_file, "r", encoding="utf-8") as f:
        lines = f.readlines()

    headings = [line.strip() for line in lines if line.startswith("#")]
    entries = [line.strip() for line in lines if line.startswith("- ")]

    print(f"File: {md_file}")
    print(f"Size: {os.path.getsize(md_file):,} bytes")
    print(f"Total lines: {len(lines):,}")
    print(f"Headings: {len(headings):,}")
    print(f"Listed studies: {len(entries):,}")
    print(f"NMDP studies: {len(nmdp_studies):,}")
    print(f"Non-NMDP studies: {len(non_nmdp_studies):,}")
    print(f"Blank lines: {sum(not line.strip() for line in lines):,}")

File: study_populations.md
Size: 2,278 bytes
Total lines: 88
Headings: 3
Listed studies: 81
NMDP studies: 20
Non-NMDP studies: 61
Blank lines: 4


In [13]:
# Compare expected NMDP populations with the populations present in the data.
# Correct the spelling variation in the dataset: "Caribean" -> "Caribbean".

observed_nmdp_groups = {
    population.removeprefix("USA NMDP ")
    .replace("Caribean", "Caribbean")
    for population in nmdp_studies
}

missing_nmdp_groups = sorted(nmdp_groups - observed_nmdp_groups)

print("Missing NMDP population(s):")
print("\n".join(missing_nmdp_groups))

Missing NMDP population(s):
Black South or Central American


In [14]:
# Add the missing expected NMDP population to study_populations.md
missing_study = "USA NMDP Black South or Central American"

with open(output_path, "r", encoding="utf-8") as file:
    content = file.read()

if f"- {missing_study}\n" not in content:
    content = content.replace(
        "## NMDP-related studies (20)",
        "## NMDP-related studies (21)"
    )
    content = content.replace(
        "## Non-NMDP studies",
        f"- {missing_study}\n\n## Non-NMDP studies"
    )

    with open(output_path, "w", encoding="utf-8") as file:
        file.write(content)

    print(f"Added '{missing_study}' to {output_path}")
else:
    print(f"'{missing_study}' is already present")

Added 'USA NMDP Black South or Central American' to study_populations.md


In [8]:
# Save to CSV
top_100_per_study.to_csv("top_100_class1.csv", index=False)
print("Saved to top_100_class1.csv")
print(f"File contains {len(top_100_per_study)} rows and {len(top_100_per_study.columns)} columns")

Saved to top_100_class1.csv
File contains 7815 rows and 7 columns


In [9]:
# Load class 2 cleaned data and get top 100 alleles per study
import os

class2_path = "cleaned_data_normal_class2.csv"
if not os.path.exists(class2_path):
    print(f"{class2_path} not found, skipping.")
else:
    df_class2 = pd.read_csv(class2_path, index_col=0)
    print(f"Loaded {class2_path}: {df_class2.shape[0]} rows, {df_class2['population'].nunique()} studies")
    print(f"Genes: {sorted(df_class2['gene'].unique())}")

    top_100_class2 = get_top_n_alleles_per_study(df_class2, n=100, verbose=True)

    top_100_class2.to_csv("top_100_class2.csv", index=False)
    print(f"\nSaved to top_100_class2.csv ({len(top_100_class2)} rows)")


Loaded cleaned_data_normal_class2.csv: 6656 rows, 86 studies
Genes: ['DPA1', 'DPB1', 'DQA1', 'DQB1', 'DRB1']
Processing 86 studies...
Getting top 100 alleles from each study

Summary:
Total studies processed: 86
Studies with < 100 alleles: 70
Studies with >= 100 alleles: 16
Total alleles in result: 5425

Result shape: (5425, 6)

Saved to top_100_class2.csv (5425 rows)
